In [ ]:
#!/usr/bin/env python
# coding: utf-8
"""
Limpieza de datos: elimina duplicados de datetime, filas con NaNs.
Divide por transecto y guarda también archivos globales por estación.
"""

import os
import numpy as np
import pandas as pd
from pathlib import Path


INPUT_DIR = os.path.expanduser("/Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/")

OUTPUT_BY_TRANSECT = os.path.join(BASE_DIR, "imputed_by_transect")
OUTPUT_GLOBAL = os.path.join(BASE_DIR, "imputed_global")
os.makedirs(OUTPUT_BY_TRANSECT, exist_ok=True)
os.makedirs(OUTPUT_GLOBAL, exist_ok=True)

NUM_COLS = ["NO", "NO2", "NOx", "O3_for_impute", "O3",
            "Veloc.", "Direc.", "Temp.", "R.Sol.", "Dist.", "Angulo"]
MAX_SHOW_PER_COLUMN = 10
MAX_SHOW_DUPLICATE_TIMESTAMPS = 20


def load_station_data(filepath):
    df = pd.read_csv(filepath, index_col=0, parse_dates=True, low_memory=False)
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index, errors="coerce")
    return df


def detect_transect_label(df):
    if "Transecto" not in df.columns:
        return None
    tran_names = df["Transecto"].dropna().unique()
    return tran_names[0] if len(tran_names) > 0 else None


def convert_numeric_columns(df):
    df = df.copy()
    for col in NUM_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def report_nans(df, label, max_show_per_column=MAX_SHOW_PER_COLUMN):
    nan_mask = df.isna()
    total_nans = int(nan_mask.sum().sum())
    nans_by_column = nan_mask.sum()
    nans_by_column = nans_by_column[nans_by_column > 0].sort_values(ascending=False)
    print(f"\n[{label}] NaNs detectados: {total_nans}")
    if total_nans == 0:
        print(f"[{label}] No hay NaNs.")
        return total_nans
    print(f"[{label}] NaNs por columna:")
    for col, n in nans_by_column.items():
        positions = df.index[nan_mask[col]].tolist()
        preview = positions[:max_show_per_column]
        preview_str = ", ".join(str(x) for x in preview)
        more = "" if len(positions) <= max_show_per_column else f" ... (+{len(positions)-max_show_per_column} más)"
        print(f"  - {col}: {int(n)} -> [{preview_str}]{more}")
    return total_nans


def report_duplicates(df, label, max_show=MAX_SHOW_DUPLICATE_TIMESTAMPS):
    dup_mask = df.index.duplicated(keep=False)
    dup_rows = int(dup_mask.sum())
    if dup_rows == 0:
        print(f"\n[{label}] No hay duplicados de datetime.")
        return {"duplicate_rows": 0, "duplicate_timestamps": 0, "duplicate_locations": []}
    duplicated_index_values = df.index[dup_mask]
    dup_counts = pd.Series(duplicated_index_values).value_counts().sort_index()
    duplicate_timestamps = int(len(dup_counts))
    print(f"\n[{label}] Duplicados de datetime detectados: filas implicadas={dup_rows}, timestamps únicos={duplicate_timestamps}")
    # Mostrar primeros
    preview_timestamps = list(dup_counts.index[:max_show])
    duplicate_locations = []
    for ts in preview_timestamps:
        pos = np.where(df.index == ts)[0].tolist()
        duplicate_locations.append((str(ts), pos))
        pos_str = ", ".join(str(p) for p in pos[:20])
        more = "" if len(pos) <= 20 else f" ... (+{len(pos)-20} más)"
        print(f"  - {ts}: posiciones [{pos_str}]{more}")
    return {"duplicate_rows": dup_rows, "duplicate_timestamps": duplicate_timestamps, "duplicate_locations": duplicate_locations}


def clean_dataframe(df, label, station_name=None, transect_clean=None):
    df = df.copy()
    # Rellenar metadatos
    if station_name is not None:
        if "Estacion" not in df.columns:
            df["Estacion"] = station_name
        else:
            df["Estacion"] = df["Estacion"].fillna(station_name)
    if transect_clean is not None and "Transecto" in df.columns:
        df["Transecto"] = df["Transecto"].fillna(transect_clean.replace("_", " "))
    # Convertir numéricas
    df = convert_numeric_columns(df)
    original_shape = df.shape
    total_nans_before = int(df.isna().sum().sum())
    duplicate_info = report_duplicates(df, label)
    nans_before = report_nans(df, label)
    # Eliminar NaT
    nat_mask = df.index.isna()
    nat_count = int(nat_mask.sum())
    if nat_count > 0:
        print(f"\n[{label}] Filas con índice NaT: {nat_count}")
        df = df.loc[~nat_mask].copy()
    # Ordenar
    df = df.sort_index(kind="mergesort")
    # Eliminar duplicados (keep first)
    dup_mask_after_nat = df.index.duplicated(keep="first")
    duplicated_rows_removed = int(dup_mask_after_nat.sum())
    if duplicated_rows_removed > 0:
        print(f"\n[{label}] Filas duplicadas eliminadas: {duplicated_rows_removed}")
    df = df.loc[~dup_mask_after_nat].copy()
    rows_after_duplicates = df.shape[0]
    # Eliminar filas con NaN
    nan_rows_mask = df.isna().any(axis=1)
    nan_rows_removed = int(nan_rows_mask.sum())
    if nan_rows_removed > 0:
        print(f"\n[{label}] Filas con NaN eliminadas: {nan_rows_removed}")
    df = df.loc[~nan_rows_mask].copy()
    final_shape = df.shape
    print(f"\n[{label}] RESUMEN: original {original_shape} -> final {final_shape}")
    return df


def prepare_station_dataframe(df, station_name, transect_clean=None):
    df = df.copy()
    if "Estacion" not in df.columns:
        df["Estacion"] = station_name
    else:
        df["Estacion"] = df["Estacion"].fillna(station_name)
    return clean_dataframe(df, station_name, station_name, transect_clean)


def finalize_combined_dataframe(df, label):
    df = df.copy()
    nat_mask = df.index.isna()
    nat_count = int(nat_mask.sum())
    if nat_count > 0:
        print(f"\n[{label}] Índices NaT tras concatenación: {nat_count}")
        df = df.loc[~nat_mask].copy()
    df = df.sort_index(kind="mergesort")
    dup_mask = df.index.duplicated(keep="first")
    dup_count = int(dup_mask.sum())
    if dup_count > 0:
        print(f"\n[{label}] Duplicados residuales eliminados: {dup_count}")
        df = df.loc[~dup_mask].copy()
    nan_rows_mask = df.isna().any(axis=1)
    nan_count = int(nan_rows_mask.sum())
    if nan_count > 0:
        print(f"[{label}] Filas con NaN residuales eliminadas: {nan_count}")
        df = df.loc[~nan_rows_mask].copy()
    return df


def process_by_transect():
    print("\n=== Limpieza por transecto ===")
    outlier_files = list(Path(INPUT_DIR).glob("*_outliers.csv"))
    if not outlier_files:
        print("No se encontraron archivos *_outliers.csv")
        return {}, {}
    transect_dict = {}
    station_to_transect = {}
    cleaned_station_data = {}
    for filepath in outlier_files:
        station_name = filepath.stem.replace("_outliers", "")
        df_raw = load_station_data(filepath)
        if "Transecto" not in df_raw.columns:
            print(f"Advertencia: {filepath.name} no tiene Transecto. Se omite.")
            continue
        transect = detect_transect_label(df_raw)
        if transect is None:
            print(f"Advertencia: {filepath.name} no tiene valores en Transecto. Se omite.")
            continue
        transect_clean = str(transect).replace(" ", "_")
        station_to_transect[station_name] = transect_clean
        df_clean = prepare_station_dataframe(df_raw, station_name, transect_clean)
        cleaned_station_data[station_name] = df_clean
        transect_dict.setdefault(transect_clean, []).append((station_name, df_clean))
    for transect_clean, station_list in transect_dict.items():
        print(f"\n=== Procesando transecto: {transect_clean} ===")
        df_concat = pd.concat([df for _, df in station_list], axis=0, sort=False)
        df_concat = finalize_combined_dataframe(df_concat, f"TRANSECTO {transect_clean}")
        original_rows = sum(df.shape[0] for _, df in station_list)
        out_path = os.path.join(OUTPUT_BY_TRANSECT, f"{transect_clean}.csv")
        df_concat.to_csv(out_path, index=True)
        print(f"  Original: {original_rows} filas, Final: {df_concat.shape[0]} filas -> Guardado {out_path}")
    return cleaned_station_data, station_to_transect


def process_global(cleaned_station_data, station_to_transect):
    print("\n=== Limpieza global (por estación) ===")
    if not cleaned_station_data:
        return
    for station_name, df_station in cleaned_station_data.items():
        df_station = df_station.copy()
        if "Estacion" not in df_station.columns:
            df_station["Estacion"] = station_name
        if "Transecto" in df_station.columns:
            df_station["Transecto"] = df_station["Transecto"].ffill().bfill()
        df_station = finalize_combined_dataframe(df_station, f"ESTACION {station_name}")
        out_path = os.path.join(OUTPUT_GLOBAL, f"{station_name}.csv")
        df_station.to_csv(out_path, index=True)
        print(f"  Guardado: {out_path} | tamaño {df_station.shape}")


if __name__ == "__main__":
    print("Iniciando limpieza de datos...")
    cleaned, mapping = process_by_transect()
    process_global(cleaned, mapping)
    print("\nProceso completado.") 



#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Codificación de variables: cíclicas, direccionales, one-hot (ML) y factorize (DL).
Se aplica a transectos y a estaciones individuales.
"""

import os
import json
import re
import numpy as np
import pandas as pd
from pathlib import Path

BASE_DIR = os.path.expanduser("/Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/")
ENCODED_DIR = os.path.join(BASE_DIR, "encoded")

INPUT_BY_TRANSECT = os.path.join(BASE_DIR, "imputed_by_transect")
INPUT_GLOBAL = os.path.join(BASE_DIR, "imputed_global")

OUTPUT_ML_TRANSECT = os.path.join(ENCODED_DIR, "ml", "by_transect")
OUTPUT_DL_TRANSECT = os.path.join(ENCODED_DIR, "dl", "by_transect")
OUTPUT_ML_GLOBAL = os.path.join(ENCODED_DIR, "ml", "global")
OUTPUT_DL_GLOBAL = os.path.join(ENCODED_DIR, "dl", "global")

for path in [OUTPUT_ML_TRANSECT, OUTPUT_DL_TRANSECT, OUTPUT_ML_GLOBAL, OUTPUT_DL_GLOBAL]:
    os.makedirs(path, exist_ok=True)

NUM_COLS = ['NO', 'NO2', 'NOx', 'O3', 'Veloc.', 'Direc.', 'Temp.', 'R.Sol.', 'Dist.', 'Angulo']
CATEGORICAL_COLS = ['Estacion', 'Transecto']


def cyclical_encode(series, period):
    rad = 2 * np.pi * series / period
    return np.sin(rad), np.cos(rad)


def standardize_target_column(df):
    df = df.copy()
    if "O3_for_impute" in df.columns and "O3" not in df.columns:
        df.rename(columns={"O3_for_impute": "O3"}, inplace=True)
    elif "O3_for_impute" in df.columns and "O3" in df.columns:
        df["O3"] = df["O3"].where(df["O3"].notna(), df["O3_for_impute"])
        df.drop(columns=["O3_for_impute"], inplace=True)
    return df


def add_datetime_features(df):
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("El índice debe ser DatetimeIndex")
    idx = df.index
    hour = idx.hour
    dayofyear = idx.dayofyear
    week = idx.isocalendar().week.astype(int)
    month = idx.month
    year = idx.year
    df['hour_sin'], df['hour_cos'] = cyclical_encode(hour, 24)
    df['day_sin'], df['day_cos'] = cyclical_encode(dayofyear, 365)
    df['week_sin'], df['week_cos'] = cyclical_encode(week, 52)
    df['month_sin'], df['month_cos'] = cyclical_encode(month, 12)
    df['year'] = year
    return df


def add_directional_features(df):
    df = df.copy()
    for col in ['Direc.', 'Angulo']:
        if col in df.columns:
            values = pd.to_numeric(df[col], errors="coerce")
            rad = np.radians(values)
            df[f'{col}sin'] = np.sin(rad)
            df[f'{col}cos'] = np.cos(rad)
            df.drop(columns=[col], inplace=True)
    return df


def normalize_station(val):
    s = str(val).strip()
    m = re.search(r'Estacion[ _]?(\d+)', s, re.IGNORECASE) or re.search(r'\bE[ _]?(\d+)\b', s, re.IGNORECASE) or re.search(r'(\d+)', s)
    return f"Estacion_{m.group(1)}" if m else s.replace(' ', '_')


def normalize_transect(val):
    s = str(val).strip()
    m = re.search(r'Transecto[ _]?(\d+)', s, re.IGNORECASE) or re.search(r'\bT[ _]?(\d+)\b', s, re.IGNORECASE) or re.search(r'(\d+)', s)
    return f"Transecto_{m.group(1)}" if m else s.replace(' ', '_')


def encode_categorical_ml(df, cat_cols):
    df = df.copy()
    existing = [c for c in cat_cols if c in df.columns]
    if not existing:
        return df
    if 'Estacion' in existing:
        df['Estacion'] = df['Estacion'].apply(normalize_station)
    if 'Transecto' in existing:
        df['Transecto'] = df['Transecto'].apply(normalize_transect)
    dummies = pd.get_dummies(df[existing].astype(str), prefix='', prefix_sep='')
    dummies.columns = [col.replace(' ', '_').replace('/', '_') for col in dummies.columns]
    df = pd.concat([df, dummies], axis=1)
    df.drop(columns=existing, inplace=True)
    return df


def encode_categorical_dl(df, cat_cols, save_mapping=True, mapping_file=None):
    df = df.copy()
    existing = [c for c in cat_cols if c in df.columns]
    if not existing:
        return df, {}
    mapping = {}
    for col in existing:
        codes, uniques = pd.factorize(df[col], sort=False)
        df[col] = codes
        mapping[col] = {str(category): int(code) for code, category in enumerate(uniques)}
    if save_mapping and mapping_file:
        with open(mapping_file, 'w', encoding='utf-8') as f:
            json.dump(mapping, f, indent=2)
    return df, mapping


def prepare_dataframe(df):
    df = df.copy()
    df = standardize_target_column(df)
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index, errors="coerce")
    df = df.loc[~df.index.isna()].copy()
    df = df.sort_index(kind="mergesort")
    return df


def process_file(input_path, output_ml_dir, output_dl_dir, is_global=False):
    base_name = Path(input_path).stem
    print(f"Procesando: {base_name} (global={is_global})")
    df = pd.read_csv(input_path, index_col=0, parse_dates=True, low_memory=False)
    df = prepare_dataframe(df)
    df = add_datetime_features(df)
    df = add_directional_features(df)
    # ML: one-hot
    df_ml = encode_categorical_ml(df.copy(), CATEGORICAL_COLS)
    # DL: factorize
    mapping_file = os.path.join(output_dl_dir, f"{base_name}_mapping.json")
    df_dl, _ = encode_categorical_dl(df.copy(), CATEGORICAL_COLS, save_mapping=True, mapping_file=mapping_file)
    df_ml.to_csv(os.path.join(output_ml_dir, f"{base_name}.csv"), index=True)
    df_dl.to_csv(os.path.join(output_dl_dir, f"{base_name}.csv"), index=True)
    print(f"  ML guardado, DL guardado")


def process_by_transect():
    if not os.path.exists(INPUT_BY_TRANSECT):
        print(f"La carpeta {INPUT_BY_TRANSECT} no existe.")
        return
    files = list(Path(INPUT_BY_TRANSECT).glob("*.csv"))
    for f in files:
        process_file(f, OUTPUT_ML_TRANSECT, OUTPUT_DL_TRANSECT, is_global=False)


def process_global():
    if not os.path.exists(INPUT_GLOBAL):
        print(f"La carpeta {INPUT_GLOBAL} no existe.")
        return
    files = list(Path(INPUT_GLOBAL).glob("*.csv"))
    for f in files:
        process_file(f, OUTPUT_ML_GLOBAL, OUTPUT_DL_GLOBAL, is_global=True)


if __name__ == "__main__":
    print("Iniciando codificación de variables...")
    process_by_transect()
    process_global()
    print("Proceso completado.")


#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Creación de ventanas deslizantes: 72h entrada, 72h salida.
Genera arrays X_ml (2D aplanado), X_dl (3D), y (72h).
"""

import os
import numpy as np
import pandas as pd
from pathlib import Path

BASE_DIR = os.path.expanduser("/Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/")
ENCODED_DIR = os.path.join(BASE_DIR, "encoded")
WINDOWS_DIR = os.path.join(BASE_DIR, "windows")

INPUT_ML_TRANSECT = os.path.join(ENCODED_DIR, "ml", "by_transect")
INPUT_DL_TRANSECT = os.path.join(ENCODED_DIR, "dl", "by_transect")
INPUT_ML_GLOBAL = os.path.join(ENCODED_DIR, "ml", "global")
INPUT_DL_GLOBAL = os.path.join(ENCODED_DIR, "dl", "global")

OUTPUT_ML_TRANSECT = os.path.join(WINDOWS_DIR, "by_transect", "ml")
OUTPUT_DL_TRANSECT = os.path.join(WINDOWS_DIR, "by_transect", "dl")
OUTPUT_ML_GLOBAL = os.path.join(WINDOWS_DIR, "global", "ml")
OUTPUT_DL_GLOBAL = os.path.join(WINDOWS_DIR, "global", "dl")

for path in [OUTPUT_ML_TRANSECT, OUTPUT_DL_TRANSECT, OUTPUT_ML_GLOBAL, OUTPUT_DL_GLOBAL]:
    os.makedirs(path, exist_ok=True)

WINDOW_IN = 72
WINDOW_OUT = 72
TARGET_COL = "O3"


def standardize_target_column(df):
    df = df.copy()
    if "O3" not in df.columns and "O3_for_impute" in df.columns:
        df = df.rename(columns={"O3_for_impute": "O3"})
    elif "O3" in df.columns and "O3_for_impute" in df.columns:
        df["O3"] = df["O3"].where(df["O3"].notna(), df["O3_for_impute"])
        df = df.drop(columns=["O3_for_impute"])
    return df


def ensure_datetime_index(df):
    df = df.copy()
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index, errors="coerce")
    df = df.loc[~df.index.isna()].copy()
    return df


def select_numeric_features(df):
    df = df.copy()
    df = df.select_dtypes(include=[np.number, "bool"]).copy()
    if df.shape[1] > 0:
        df = df.apply(pd.to_numeric, errors="coerce")
    return df


def create_windows_with_timestamps(df, target_col=TARGET_COL, window_in=WINDOW_IN, window_out=WINDOW_OUT):
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("DataFrame debe tener índice DatetimeIndex")
    df_sorted = df.sort_index(kind="mergesort")
    data = df_sorted.values
    feature_names = df_sorted.columns.tolist()
    if target_col not in df_sorted.columns:
        raise KeyError(f"No existe columna '{target_col}'")
    n = len(df_sorted)
    X_ml_list = []
    X_dl_list = []
    y_list = []
    timestamps = []
    last_i = n - window_out
    if last_i < window_in:
        return None, None, None, None, feature_names
    for i in range(window_in, last_i + 1):
        in_data = data[i - window_in:i, :]          # (window_in, n_features)
        out_data = df_sorted[target_col].iloc[i:i + window_out].values
        X_dl_list.append(in_data)
        X_ml_list.append(in_data.flatten())
        y_list.append(out_data)
        timestamps.append(df_sorted.index[i])
    if len(X_ml_list) == 0:
        return None, None, None, None, feature_names
    X_ml = np.array(X_ml_list, dtype=np.float32)
    X_dl = np.array(X_dl_list, dtype=np.float32)   # ya es 3D
    y = np.array(y_list, dtype=np.float32)
    timestamps = np.array(timestamps, dtype="datetime64[h]")
    return X_ml, X_dl, y, timestamps, feature_names


def process_file_ml_dl(ml_path, dl_path, output_ml_dir, output_dl_dir, name):
    print(f"  Procesando {name}...")
    df_ml = pd.read_csv(ml_path, index_col=0, parse_dates=True, low_memory=False)
    df_dl = pd.read_csv(dl_path, index_col=0, parse_dates=True, low_memory=False)
    df_ml = ensure_datetime_index(df_ml)
    df_dl = ensure_datetime_index(df_dl)
    df_ml = standardize_target_column(df_ml)
    df_dl = standardize_target_column(df_dl)
    if not df_ml.index.equals(df_dl.index):
        common_index = df_ml.index[df_ml.index.isin(df_dl.index)]
        if len(common_index) == 0:
            print(f"    Error: no hay timestamps comunes.")
            return
        df_ml = df_ml.loc[common_index].copy()
        df_dl = df_dl.loc[common_index].copy()
    df_ml = select_numeric_features(df_ml)
    df_dl = select_numeric_features(df_dl)
    if df_ml.shape[1] == 0 or df_dl.shape[1] == 0:
        print(f"    Error: sin características numéricas.")
        return
    result_ml = create_windows_with_timestamps(df_ml)
    if result_ml[0] is None:
        print(f"    No se generaron ventanas ML.")
        return
    X_ml, _, y, timestamps, feat_names_ml = result_ml
    result_dl = create_windows_with_timestamps(df_dl)
    if result_dl[0] is None:
        print(f"    No se generaron ventanas DL.")
        return
    X_dl, _, _, _, _ = result_dl
    if len(X_ml) != len(X_dl):
        print(f"    Error: número de ventanas difiere.")
        return
    # Asegurar que X_dl sea 3D (por si acaso)
    if X_dl.ndim == 2:
        n_samples = X_dl.shape[0]
        n_features = X_dl.shape[1] // WINDOW_IN
        X_dl = X_dl.reshape(n_samples, WINDOW_IN, n_features)
        print(f"    Remodelado X_dl a {X_dl.shape}")
    np.save(os.path.join(output_ml_dir, f"{name}_X.npy"), X_ml)
    np.save(os.path.join(output_ml_dir, f"{name}_y.npy"), y)
    np.save(os.path.join(output_ml_dir, f"{name}_timestamps.npy"), timestamps)
    np.save(os.path.join(output_dl_dir, f"{name}_X.npy"), X_dl)
    np.save(os.path.join(output_dl_dir, f"{name}_y.npy"), y)
    np.save(os.path.join(output_dl_dir, f"{name}_timestamps.npy"), timestamps)
    print(f"    Ventanas guardadas: {len(X_ml)} muestras. Shapes: X_ml {X_ml.shape}, X_dl {X_dl.shape}")


def process_by_transect():
    print("\n--- Procesando datos por transecto ---")
    if not os.path.exists(INPUT_ML_TRANSECT) or not os.path.exists(INPUT_DL_TRANSECT):
        print("Carpetas de entrada no existen.")
        return
    ml_files = {f.stem: f for f in Path(INPUT_ML_TRANSECT).glob("*.csv")}
    dl_files = {f.stem: f for f in Path(INPUT_DL_TRANSECT).glob("*.csv")}
    common = set(ml_files.keys()) & set(dl_files.keys())
    for name in sorted(common):
        process_file_ml_dl(ml_files[name], dl_files[name], OUTPUT_ML_TRANSECT, OUTPUT_DL_TRANSECT, name)


def process_global():
    print("\n--- Procesando datos globales (por estación) ---")
    if not os.path.exists(INPUT_ML_GLOBAL) or not os.path.exists(INPUT_DL_GLOBAL):
        print("Carpetas de entrada no existen.")
        return
    ml_files = {f.stem: f for f in Path(INPUT_ML_GLOBAL).glob("*.csv")}
    dl_files = {f.stem: f for f in Path(INPUT_DL_GLOBAL).glob("*.csv")}
    common = set(ml_files.keys()) & set(dl_files.keys())
    for name in sorted(common):
        process_file_ml_dl(ml_files[name], dl_files[name], OUTPUT_ML_GLOBAL, OUTPUT_DL_GLOBAL, name)


if __name__ == "__main__":
    print("Creación de ventanas deslizantes (72h in / 72h out)")
    process_by_transect()
    process_global()
    print("Proceso completado.")


#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Partición temporal (train/val/test) y normalización MinMax (solo para DL).
Solo guarda entidades donde todos los splits tengan al menos una muestra.
"""

import os
import json
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler

BASE_DIR = os.path.expanduser("/Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/")
ENCODED_DIR = os.path.join(BASE_DIR, "encoded")
WINDOWS_DIR = os.path.join(BASE_DIR, "windows")
WINDOWS_PARTITIONED_DIR = os.path.join(BASE_DIR, "windows_partitioned")

INPUT_ML_TRANSECT = os.path.join(WINDOWS_DIR, "by_transect", "ml")
INPUT_DL_TRANSECT = os.path.join(WINDOWS_DIR, "by_transect", "dl")
INPUT_ML_GLOBAL = os.path.join(WINDOWS_DIR, "global", "ml")
INPUT_DL_GLOBAL = os.path.join(WINDOWS_DIR, "global", "dl")

OUTPUT_ML_TRANSECT = os.path.join(WINDOWS_PARTITIONED_DIR, "by_transect", "ml")
OUTPUT_DL_TRANSECT = os.path.join(WINDOWS_PARTITIONED_DIR, "by_transect", "dl")
OUTPUT_ML_GLOBAL = os.path.join(WINDOWS_PARTITIONED_DIR, "global", "ml")
OUTPUT_DL_GLOBAL = os.path.join(WINDOWS_PARTITIONED_DIR, "global", "dl")

for path in [OUTPUT_ML_TRANSECT, OUTPUT_DL_TRANSECT, OUTPUT_ML_GLOBAL, OUTPUT_DL_GLOBAL]:
    os.makedirs(path, exist_ok=True)

WINDOW_IN = 72
WINDOW_OUT = 72
TARGET_COL = "O3"

# Fechas de corte (sin márgenes adicionales)
TRAIN_START = pd.to_datetime("2018-01-01 00:00:00")
TRAIN_END   = pd.to_datetime("2023-12-28 23:00:00")
VAL_START   = pd.to_datetime("2024-01-03 00:00:00")
VAL_END     = pd.to_datetime("2024-12-28 23:00:00")
TEST_START  = pd.to_datetime("2025-01-03 00:00:00")
INPUT_MARGIN = pd.Timedelta(hours=0)
OUTPUT_MARGIN = pd.Timedelta(hours=0)


def load_entity_data(entity_dir, entity_name):
    X_path = os.path.join(entity_dir, f"{entity_name}_X.npy")
    y_path = os.path.join(entity_dir, f"{entity_name}_y.npy")
    ts_path = os.path.join(entity_dir, f"{entity_name}_timestamps.npy")
    if not (os.path.exists(X_path) and os.path.exists(y_path) and os.path.exists(ts_path)):
        return None, None, None
    X = np.load(X_path)
    y = np.load(y_path)
    timestamps = pd.to_datetime(np.load(ts_path))
    return X, y, timestamps


def split_by_timestamps(X, y, timestamps):
    train_mask = (timestamps >= (TRAIN_START + INPUT_MARGIN)) & (timestamps <= (TRAIN_END - OUTPUT_MARGIN))
    val_mask   = (timestamps >= (VAL_START + INPUT_MARGIN)) & (timestamps <= (VAL_END - OUTPUT_MARGIN))
    test_mask  = timestamps >= (TEST_START + INPUT_MARGIN)
    # Resolver solapamientos
    val_mask = val_mask & ~train_mask
    test_mask = test_mask & ~train_mask & ~val_mask
    return (X[train_mask], y[train_mask]), (X[val_mask], y[val_mask]), (X[test_mask], y[test_mask])


def _scale_X_split(scaler_X, X_split, win_in, n_feat, fit=False):
    if X_split.shape[0] == 0:
        return np.empty((0, win_in, n_feat), dtype=np.float32)
    X_flat = X_split.reshape(-1, n_feat)
    if fit:
        X_scaled = scaler_X.fit_transform(X_flat)
    else:
        X_scaled = scaler_X.transform(X_flat)
    return X_scaled.reshape(X_split.shape[0], win_in, n_feat).astype(np.float32)


def _scale_y_split(scaler_y, y_split, fit=False):
    if y_split.shape[0] == 0:
        return np.empty((0, WINDOW_OUT), dtype=np.float32)
    y_flat = y_split.reshape(-1, 1)
    if fit:
        y_scaled = scaler_y.fit_transform(y_flat)
    else:
        y_scaled = scaler_y.transform(y_flat)
    return y_scaled.reshape(y_split.shape[0], WINDOW_OUT).astype(np.float32)


def normalize_data(X_train, X_val, X_test, y_train, y_val, y_test):
    n_train, win_in, n_feat = X_train.shape
    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()
    X_train_sc = _scale_X_split(scaler_X, X_train, win_in, n_feat, fit=True)
    X_val_sc   = _scale_X_split(scaler_X, X_val,   win_in, n_feat, fit=False)
    X_test_sc  = _scale_X_split(scaler_X, X_test,  win_in, n_feat, fit=False)
    y_train_sc = _scale_y_split(scaler_y, y_train, fit=True)
    y_val_sc   = _scale_y_split(scaler_y, y_val,   fit=False)
    y_test_sc  = _scale_y_split(scaler_y, y_test,  fit=False)
    return X_train_sc, X_val_sc, X_test_sc, y_train_sc, y_val_sc, y_test_sc, scaler_X, scaler_y


def save_split_ml(output_dir, entity_name, X_train, y_train, X_val, y_val, X_test, y_test):
    save_dir = os.path.join(output_dir, entity_name)
    os.makedirs(save_dir, exist_ok=True)
    np.save(os.path.join(save_dir, "train_X.npy"), X_train)
    np.save(os.path.join(save_dir, "train_y.npy"), y_train)
    np.save(os.path.join(save_dir, "val_X.npy"), X_val)
    np.save(os.path.join(save_dir, "val_y.npy"), y_val)
    np.save(os.path.join(save_dir, "test_X.npy"), X_test)
    np.save(os.path.join(save_dir, "test_y.npy"), y_test)
    print(f"    ML guardado: train={len(X_train)}, val={len(X_val)}, test={len(X_test)}")


def save_split_dl(output_dir, entity_name, X_train, y_train, X_val, y_val, X_test, y_test, scaler_X, scaler_y):
    save_dir = os.path.join(output_dir, entity_name)
    os.makedirs(save_dir, exist_ok=True)
    np.save(os.path.join(save_dir, "train_X.npy"), X_train)
    np.save(os.path.join(save_dir, "train_y.npy"), y_train)
    np.save(os.path.join(save_dir, "val_X.npy"), X_val)
    np.save(os.path.join(save_dir, "val_y.npy"), y_val)
    np.save(os.path.join(save_dir, "test_X.npy"), X_test)
    np.save(os.path.join(save_dir, "test_y.npy"), y_test)
    with open(os.path.join(save_dir, "scaler_X.pkl"), "wb") as f:
        pickle.dump(scaler_X, f)
    with open(os.path.join(save_dir, "scaler_y.pkl"), "wb") as f:
        pickle.dump(scaler_y, f)
    print(f"    DL guardado: train={len(X_train)}, val={len(X_val)}, test={len(X_test)}")


def process_entity(entity_name, input_ml_dir, input_dl_dir, output_ml_dir, output_dl_dir):
    print(f"\n  Procesando {entity_name}...")
    X_ml, y_ml, ts_ml = load_entity_data(input_ml_dir, entity_name)
    if X_ml is None:
        return
    X_dl, y_dl, ts_dl = load_entity_data(input_dl_dir, entity_name)
    if X_dl is None:
        return
    
    # Verificar que los timestamps coincidan
    if not np.array_equal(ts_ml, ts_dl):
        print(f"    Error: timestamps no coinciden. Se omite.")
        return

    # Calcular máscaras una sola vez
    train_mask = (ts_ml >= (TRAIN_START + INPUT_MARGIN)) & (ts_ml <= (TRAIN_END - OUTPUT_MARGIN))
    val_mask   = (ts_ml >= (VAL_START + INPUT_MARGIN)) & (ts_ml <= (VAL_END - OUTPUT_MARGIN))
    test_mask  = ts_ml >= (TEST_START + INPUT_MARGIN)
    val_mask = val_mask & ~train_mask
    test_mask = test_mask & ~train_mask & ~val_mask

    print(f"    Train mask sum: {train_mask.sum()}")
    print(f"    Val mask sum:   {val_mask.sum()}")
    print(f"    Test mask sum:  {test_mask.sum()}")

    # Aplicar máscaras a ML
    X_train_ml = X_ml[train_mask]
    y_train_ml = y_ml[train_mask]
    X_val_ml   = X_ml[val_mask]
    y_val_ml   = y_ml[val_mask]
    X_test_ml  = X_ml[test_mask]
    y_test_ml  = y_ml[test_mask]

    # Aplicar las MISMAS máscaras a DL
    X_train_dl = X_dl[train_mask]
    y_train_dl = y_dl[train_mask]
    X_val_dl   = X_dl[val_mask]
    y_val_dl   = y_dl[val_mask]
    X_test_dl  = X_dl[test_mask]
    y_test_dl  = y_dl[test_mask]

    # Verificar que todos los splits tengan al menos una muestra
    if len(X_train_ml) == 0 or len(X_val_ml) == 0 or len(X_test_ml) == 0:
        print(f"    ⚠️ Saltando {entity_name}: algún split vacío (train={len(X_train_ml)}, val={len(X_val_ml)}, test={len(X_test_ml)})")
        return

    # Guardar ML
    save_split_ml(output_ml_dir, entity_name, X_train_ml, y_train_ml, X_val_ml, y_val_ml, X_test_ml, y_test_ml)
    
    # Normalizar y guardar DL con el orden correcto
    if len(X_train_dl) > 0:
        X_train_sc, X_val_sc, X_test_sc, y_train_sc, y_val_sc, y_test_sc, scaler_X, scaler_y = normalize_data(
            X_train_dl, X_val_dl, X_test_dl, y_train_dl, y_val_dl, y_test_dl)
        save_split_dl(output_dl_dir, entity_name, 
                      X_train_sc, y_train_sc, 
                      X_val_sc, y_val_sc, 
                      X_test_sc, y_test_sc, 
                      scaler_X, scaler_y)
    else:
        print(f"    No hay datos DL para {entity_name}")

def process_by_transect():
    print("\n--- Procesando por transecto ---")
    ml_files = [f.stem.replace("_X", "") for f in Path(INPUT_ML_TRANSECT).glob("*_X.npy")]
    dl_files = [f.stem.replace("_X", "") for f in Path(INPUT_DL_TRANSECT).glob("*_X.npy")]
    for entity in sorted(set(ml_files) & set(dl_files)):
        process_entity(entity, INPUT_ML_TRANSECT, INPUT_DL_TRANSECT, OUTPUT_ML_TRANSECT, OUTPUT_DL_TRANSECT)


def process_global():
    print("\n--- Procesando global ---")
    ml_files = [f.stem.replace("_X", "") for f in Path(INPUT_ML_GLOBAL).glob("*_X.npy")]
    dl_files = [f.stem.replace("_X", "") for f in Path(INPUT_DL_GLOBAL).glob("*_X.npy")]
    for entity in sorted(set(ml_files) & set(dl_files)):
        process_entity(entity, INPUT_ML_GLOBAL, INPUT_DL_GLOBAL, OUTPUT_ML_GLOBAL, OUTPUT_DL_GLOBAL)


if __name__ == "__main__":
    print("Partición temporal + normalización")
    process_by_transect()
    process_global()
    print("Proceso completado.")


#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Preparación final: crea versiones 2D aplanadas (ml_2d) para algoritmos tipo árbol.
Solo se generan si todos los splits tienen al menos una muestra.
"""

import os
import json
import numpy as np
from pathlib import Path

BASE_DIR = os.path.expanduser("/Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/")
WINDOWS_PARTITIONED_DIR = os.path.join(BASE_DIR, "windows_partitioned")

INPUT_DIR = WINDOWS_PARTITIONED_DIR
PATHS = {
    "by_transect_ml": os.path.join(INPUT_DIR, "by_transect", "ml"),
    "by_transect_dl": os.path.join(INPUT_DIR, "by_transect", "dl"),
    "global_ml": os.path.join(INPUT_DIR, "global", "ml"),
    "global_dl": os.path.join(INPUT_DIR, "global", "dl"),
}
WINDOW_IN = 72


def ensure_2d(X):
    if X.ndim == 3:
        n_samples, win_in, n_features = X.shape
        return X.reshape(n_samples, win_in * n_features)
    elif X.ndim == 2:
        return X
    else:
        raise ValueError(f"Dimensiones inesperadas: {X.ndim}")


def process_entity_ml(ml_dir, entity_name):
    entity_path = os.path.join(ml_dir, entity_name)
    if not os.path.isdir(entity_path):
        return False
    files = {
        'train_X': os.path.join(entity_path, "train_X.npy"),
        'val_X': os.path.join(entity_path, "val_X.npy"),
        'test_X': os.path.join(entity_path, "test_X.npy"),
        'train_y': os.path.join(entity_path, "train_y.npy"),
        'val_y': os.path.join(entity_path, "val_y.npy"),
        'test_y': os.path.join(entity_path, "test_y.npy"),
    }
    for name, path in files.items():
        if not os.path.exists(path):
            print(f"    Falta {name} para {entity_name}, se omite.")
            return False

    train_X = np.load(files['train_X'])
    val_X   = np.load(files['val_X'])
    test_X  = np.load(files['test_X'])
    train_y = np.load(files['train_y'])
    val_y   = np.load(files['val_y'])
    test_y  = np.load(files['test_y'])

    # Verificar que ningún split esté vacío
    if len(train_X) == 0 or len(val_X) == 0 or len(test_X) == 0:
        print(f"    Saltando {entity_name}: split vacío (train={len(train_X)}, val={len(val_X)}, test={len(test_X)})")
        return False

    train_X_2d = ensure_2d(train_X)
    val_X_2d   = ensure_2d(val_X)
    test_X_2d  = ensure_2d(test_X)

    out_dir = os.path.join(entity_path, "ml_2d")
    os.makedirs(out_dir, exist_ok=True)

    np.save(os.path.join(out_dir, "train_X.npy"), train_X_2d)
    np.save(os.path.join(out_dir, "val_X.npy"),   val_X_2d)
    np.save(os.path.join(out_dir, "test_X.npy"),  test_X_2d)
    np.save(os.path.join(out_dir, "train_y.npy"), train_y)
    np.save(os.path.join(out_dir, "val_y.npy"),   val_y)
    np.save(os.path.join(out_dir, "test_y.npy"),  test_y)

    print(f"    ML 2D generado para {entity_name}: {train_X_2d.shape}")
    return True


def process_ml_folder(ml_dir, description):
    if not os.path.exists(ml_dir):
        print(f"  No existe {ml_dir}")
        return
    entities = [d for d in os.listdir(ml_dir) if os.path.isdir(os.path.join(ml_dir, d))]
    if not entities:
        print(f"  No hay entidades en {ml_dir}")
        return
    print(f"\n--- Procesando {description} ---")
    for entity in sorted(entities):
        process_entity_ml(ml_dir, entity)


def generate_metadata():
    metadata = {
        "by_transect": {"ml": {}, "dl": {}},
        "global": {"ml": {}, "dl": {}}
    }
    def get_shapes(base_dir, subdir, entity):
        shapes = {}
        for split in ['train', 'val', 'test']:
            X_path = os.path.join(base_dir, subdir, entity, f"{split}_X.npy")
            y_path = os.path.join(base_dir, subdir, entity, f"{split}_y.npy")
            if os.path.exists(X_path):
                X = np.load(X_path, mmap_mode='r')
                shapes[f"{split}_X"] = list(X.shape)
            if os.path.exists(y_path):
                y = np.load(y_path, mmap_mode='r')
                shapes[f"{split}_y"] = list(y.shape)
        return shapes
    # by_transect
    for sub in ['ml', 'dl']:
        base = PATHS[f"by_transect_{sub}"]
        if os.path.exists(base):
            for entity in os.listdir(base):
                if os.path.isdir(os.path.join(base, entity)):
                    metadata["by_transect"][sub][entity] = get_shapes(PATHS["by_transect_ml"] if sub=='ml' else PATHS["by_transect_dl"], sub, entity)
                    if sub == 'ml':
                        ml_2d_path = os.path.join(base, entity, "ml_2d")
                        if os.path.exists(ml_2d_path):
                            metadata["by_transect"]["ml"][entity]["ml_2d"] = get_shapes(base, entity, "ml_2d")
    # global
    for sub in ['ml', 'dl']:
        base = PATHS[f"global_{sub}"]
        if os.path.exists(base):
            for entity in os.listdir(base):
                if os.path.isdir(os.path.join(base, entity)):
                    metadata["global"][sub][entity] = get_shapes(PATHS["global_ml"] if sub=='ml' else PATHS["global_dl"], sub, entity)
                    if sub == 'ml':
                        ml_2d_path = os.path.join(base, entity, "ml_2d")
                        if os.path.exists(ml_2d_path):
                            metadata["global"]["ml"][entity]["ml_2d"] = get_shapes(base, entity, "ml_2d")
    meta_path = os.path.join(INPUT_DIR, "dataset_metadata.json")
    with open(meta_path, 'w') as f:
        json.dump(metadata, f, indent=2)
    print(f"\nMetadatos guardados en {meta_path}")


if __name__ == "__main__":
    print("Preparación final de datasets (ML 2D)")
    process_ml_folder(PATHS["by_transect_ml"], "datos por transecto (ML)")
    process_ml_folder(PATHS["global_ml"], "datos globales (ML)")
    generate_metadata()
    print("Proceso completado.")


#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Random Forest con búsqueda de hiperparámetros.
Solo entrena si hay datos de validación.
"""

import os
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

BASE_DIR = os.path.expanduser("/Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/")
WINDOWS_PARTITIONED_DIR = os.path.join(BASE_DIR, "windows_partitioned")
MODELS_DIR = os.path.join(BASE_DIR, "models")
ENCODED_DIR = os.path.join(BASE_DIR, "encoded")  # para cargar CSV originales (necesario para estaciones)

DATA_DIR = WINDOWS_PARTITIONED_DIR
OUTPUT_DIR = os.path.join(MODELS_DIR, "random_forest")
os.makedirs(OUTPUT_DIR, exist_ok=True)

WINDOW_IN = 72
WINDOW_OUT = 72
RANDOM_STATE = 42
N_JOBS = -1

RF_PARAM_GRID = {
    'n_estimators': [500],
    'max_depth': [30, 40],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [2, 4],
    'max_features': ['sqrt', 'log2']
}


def willmott_index(y_true, y_pred):
    numer = np.sum((y_true - y_pred) ** 2)
    denom = np.sum((np.abs(y_pred - y_true.mean()) + np.abs(y_true - y_true.mean())) ** 2)
    return 1 - numer / denom if denom != 0 else np.nan


def mape(y_true, y_pred):
    mask = y_true != 0
    if not mask.any():
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


def compute_metrics(y_true, y_pred):
    return {
        'r2': r2_score(y_true, y_pred),
        'mae': mean_absolute_error(y_true, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'mape': mape(y_true, y_pred),
        'willmott': willmott_index(y_true, y_pred)
    }


def plot_predictions(y_true, y_pred, horizons, save_path, title):
    n_plots = len(horizons)
    fig, axes = plt.subplots(1, n_plots, figsize=(5*n_plots, 4))
    if n_plots == 1:
        axes = [axes]
    for ax, h in zip(axes, horizons):
        ax.scatter(y_true[:, h], y_pred[:, h], alpha=0.3, s=10)
        ax.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', lw=1)
        ax.set_xlabel('Real O3 (µg/m³)')
        ax.set_ylabel('Predicho O3 (µg/m³)')
        ax.set_title(f'Horizonte {h+1}h')
        ax.grid(True, alpha=0.3)
    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()


def get_station_from_features(X_sample, feature_names, station_prefix='Estacion_'):
    n_features = len(feature_names)
    first_block = X_sample[:n_features]
    station_indices = [i for i, name in enumerate(feature_names) if name.startswith(station_prefix)]
    for idx in station_indices:
        if abs(first_block[idx] - 1.0) < 0.1:
            station_name = feature_names[idx][len(station_prefix):]
            return station_name
    return None


def compute_per_station_metrics(y_true, y_pred, X_test, feature_names, output_dir, entity_name):
    n_samples = len(y_true)
    station_preds = {}
    for i in range(n_samples):
        station = get_station_from_features(X_test[i], feature_names)
        if station is None:
            continue
        station_preds.setdefault(station, {'true': [], 'pred': []})
        station_preds[station]['true'].append(y_true[i])
        station_preds[station]['pred'].append(y_pred[i])
    station_metrics = {}
    for station, data in station_preds.items():
        true_stack = np.vstack(data['true'])
        pred_stack = np.vstack(data['pred'])
        metrics = compute_metrics(true_stack.ravel(), pred_stack.ravel())
        station_metrics[station] = metrics
    if station_metrics:
        df = pd.DataFrame(station_metrics).T
        df.index.name = 'station'
        df.to_csv(os.path.join(output_dir, f"{entity_name}_per_station_metrics.csv"))
        print(f"    Métricas por estación guardadas.")
    return station_metrics


def train_and_evaluate_rf(X_train, y_train, X_val, X_test, y_val, y_test,
                          entity_name, output_subdir, feature_names=None, original_csv_path=None):
    print(f"\n--- Entrenando Random Forest para {entity_name} ---")
    if len(X_val) == 0 or len(y_val) == 0:
        print(f"  Saltando {entity_name}: sin datos de validación.")
        return None, None

    tscv = TimeSeriesSplit(n_splits=3)
    rf = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=1)
    grid_search = GridSearchCV(estimator=rf, param_grid=RF_PARAM_GRID, cv=tscv, scoring='r2', n_jobs=N_JOBS, verbose=2)
    print("  Buscando mejores hiperparámetros...")
    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_
    print(f"  Mejores parámetros: {grid_search.best_params_}")

    val_pred = best_model.predict(X_val)
    val_metrics = compute_metrics(y_val.ravel(), val_pred.ravel())
    print(f"  Métricas en validación: R2={val_metrics['r2']:.3f}, MAE={val_metrics['mae']:.2f}")

    test_pred = best_model.predict(X_test)
    test_metrics = compute_metrics(y_test.ravel(), test_pred.ravel())
    print(f"  Métricas en test: R2={test_metrics['r2']:.3f}, MAE={test_metrics['mae']:.2f}, RMSE={test_metrics['rmse']:.2f}")

    if feature_names is not None and original_csv_path is not None:
        station_metrics = compute_per_station_metrics(y_test, test_pred, X_test, feature_names, output_subdir, entity_name)
    else:
        station_metrics = None

    with open(os.path.join(output_subdir, "model.pkl"), 'wb') as f:
        pickle.dump(best_model, f)
    results = {'best_params': grid_search.best_params_, 'validation_metrics': val_metrics,
               'test_metrics': test_metrics, 'station_metrics': station_metrics,
               'n_train': len(X_train), 'n_val': len(X_val), 'n_test': len(X_test)}
    with open(os.path.join(output_subdir, "results.json"), 'w') as f:
        json.dump(results, f, indent=2)

    horizons = [23, 47, 71]
    plot_predictions(y_test, test_pred, horizons, os.path.join(output_subdir, "test_scatter.png"),
                     f"Random Forest - {entity_name} - Test")
    np.save(os.path.join(output_subdir, "test_pred.npy"), test_pred)
    np.save(os.path.join(output_subdir, "test_true.npy"), y_test)
    return best_model, test_metrics


def process_by_transect():
    print("\n" + "="*50)
    print("PROCESANDO RF POR TRANSECTO")
    print("="*50)
    ml_2d_dir = os.path.join(DATA_DIR, "by_transect", "ml")
    if not os.path.exists(ml_2d_dir):
        return
    entities = [d for d in os.listdir(ml_2d_dir) if os.path.isdir(os.path.join(ml_2d_dir, d))]
    for entity in entities:
        ml_2d_path = os.path.join(ml_2d_dir, entity, "ml_2d")
        if not os.path.exists(ml_2d_path):
            continue
        X_train = np.load(os.path.join(ml_2d_path, "train_X.npy"))
        y_train = np.load(os.path.join(ml_2d_path, "train_y.npy"))
        X_val   = np.load(os.path.join(ml_2d_path, "val_X.npy"))
        y_val   = np.load(os.path.join(ml_2d_path, "val_y.npy"))
        X_test  = np.load(os.path.join(ml_2d_path, "test_X.npy"))
        y_test  = np.load(os.path.join(ml_2d_path, "test_y.npy"))

        original_csv = os.path.join(ENCODED_DIR, "ml", "by_transect", f"{entity}.csv")
        feature_names = None
        if os.path.exists(original_csv):
            df_cols = pd.read_csv(original_csv, nrows=0, index_col=0)
            feature_names = df_cols.columns.tolist()

        out_subdir = os.path.join(OUTPUT_DIR, "by_transect", entity)
        os.makedirs(out_subdir, exist_ok=True)
        train_and_evaluate_rf(X_train, y_train, X_val, X_test, y_val, y_test,
                              entity, out_subdir, feature_names, original_csv if os.path.exists(original_csv) else None)


def process_global():
    print("\n" + "="*50)
    print("PROCESANDO RF GLOBAL")
    print("="*50)
    ml_2d_dir = os.path.join(DATA_DIR, "global", "ml")
    if not os.path.exists(ml_2d_dir):
        return
    entities = [d for d in os.listdir(ml_2d_dir) if os.path.isdir(os.path.join(ml_2d_dir, d))]
    for entity in entities:
        ml_2d_path = os.path.join(ml_2d_dir, entity, "ml_2d")
        if not os.path.exists(ml_2d_path):
            continue
        X_train = np.load(os.path.join(ml_2d_path, "train_X.npy"))
        y_train = np.load(os.path.join(ml_2d_path, "train_y.npy"))
        X_val   = np.load(os.path.join(ml_2d_path, "val_X.npy"))
        y_val   = np.load(os.path.join(ml_2d_path, "val_y.npy"))
        X_test  = np.load(os.path.join(ml_2d_path, "test_X.npy"))
        y_test  = np.load(os.path.join(ml_2d_path, "test_y.npy"))

        out_subdir = os.path.join(OUTPUT_DIR, "global", entity)
        os.makedirs(out_subdir, exist_ok=True)
        train_and_evaluate_rf(X_train, y_train, X_val, X_test, y_val, y_test,
                              entity, out_subdir, feature_names=None, original_csv_path=None)


def generate_summary():
    summary = []
    for t in ["by_transect", "global"]:
        dir_path = os.path.join(OUTPUT_DIR, t)
        if not os.path.exists(dir_path):
            continue
        for entity in os.listdir(dir_path):
            res_file = os.path.join(dir_path, entity, "results.json")
            if os.path.exists(res_file):
                with open(res_file, 'r') as f:
                    data = json.load(f)
                summary.append({'entity': entity, 'type': t, **data['test_metrics']})
    if summary:
        pd.DataFrame(summary).to_csv(os.path.join(OUTPUT_DIR, "summary_metrics.csv"), index=False)
        print("\nResumen guardado.")


if __name__ == "__main__":
    print("RANDOM FOREST")
    process_by_transect()
    process_global()
    generate_summary()


#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
XGBoost con MultiOutputRegressor y búsqueda de hiperparámetros.
"""

import os
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

BASE_DIR = os.path.expanduser("/Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/")
WINDOWS_PARTITIONED_DIR = os.path.join(BASE_DIR, "windows_partitioned")
MODELS_DIR = os.path.join(BASE_DIR, "models")
ENCODED_DIR = os.path.join(BASE_DIR, "encoded")  # para cargar CSV originales (necesario para estaciones)

DATA_DIR = WINDOWS_PARTITIONED_DIR
OUTPUT_DIR = os.path.join(MODELS_DIR, "xgboost")
os.makedirs(OUTPUT_DIR, exist_ok=True)

WINDOW_IN = 72
WINDOW_OUT = 72
RANDOM_STATE = 42
N_JOBS = -1

XGB_PARAM_GRID = {
    'estimator__n_estimators': [150, 250],
    'estimator__max_depth': [5, 8],
    'estimator__learning_rate': [0.05, 0.1],
    'estimator__subsample': [0.8, 1.0],
    'estimator__colsample_bytree': [0.8, 1.0],
    'estimator__reg_alpha': [0, 0.1],
    'estimator__reg_lambda': [1, 1.5]
}


def willmott_index(y_true, y_pred):
    numer = np.sum((y_true - y_pred) ** 2)
    denom = np.sum((np.abs(y_pred - y_true.mean()) + np.abs(y_true - y_true.mean())) ** 2)
    return 1 - numer / denom if denom != 0 else np.nan


def mape(y_true, y_pred):
    mask = y_true != 0
    if not mask.any():
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


def compute_metrics(y_true, y_pred):
    return {
        'r2': r2_score(y_true, y_pred),
        'mae': mean_absolute_error(y_true, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'mape': mape(y_true, y_pred),
        'willmott': willmott_index(y_true, y_pred)
    }


def plot_predictions(y_true, y_pred, horizons, save_path, title):
    n_plots = len(horizons)
    fig, axes = plt.subplots(1, n_plots, figsize=(5*n_plots, 4))
    if n_plots == 1:
        axes = [axes]
    for ax, h in zip(axes, horizons):
        ax.scatter(y_true[:, h], y_pred[:, h], alpha=0.3, s=10)
        ax.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', lw=1)
        ax.set_xlabel('Real O3 (µg/m³)')
        ax.set_ylabel('Predicho O3 (µg/m³)')
        ax.set_title(f'Horizonte {h+1}h')
        ax.grid(True, alpha=0.3)
    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()


def get_station_from_features(X_sample, feature_names, station_prefix='Estacion_'):
    n_features = len(feature_names)
    first_block = X_sample[:n_features]
    station_indices = [i for i, name in enumerate(feature_names) if name.startswith(station_prefix)]
    for idx in station_indices:
        if abs(first_block[idx] - 1.0) < 0.1:
            return feature_names[idx][len(station_prefix):]
    return None


def compute_per_station_metrics(y_true, y_pred, X_test, feature_names, output_dir, entity_name):
    n_samples = len(y_true)
    station_preds = {}
    for i in range(n_samples):
        station = get_station_from_features(X_test[i], feature_names)
        if station is None:
            continue
        station_preds.setdefault(station, {'true': [], 'pred': []})
        station_preds[station]['true'].append(y_true[i])
        station_preds[station]['pred'].append(y_pred[i])
    station_metrics = {}
    for station, data in station_preds.items():
        true_stack = np.vstack(data['true'])
        pred_stack = np.vstack(data['pred'])
        metrics = compute_metrics(true_stack.ravel(), pred_stack.ravel())
        station_metrics[station] = metrics
    if station_metrics:
        df = pd.DataFrame(station_metrics).T
        df.index.name = 'station'
        df.to_csv(os.path.join(output_dir, f"{entity_name}_per_station_metrics.csv"))
    return station_metrics


def train_and_evaluate_xgb(X_train, y_train, X_val, X_test, y_val, y_test,
                           entity_name, output_subdir, feature_names=None, original_csv_path=None):
    print(f"\n--- Entrenando XGBoost para {entity_name} ---")
    if len(X_val) == 0 or len(y_val) == 0:
        print(f"  Saltando {entity_name}: sin datos de validación.")
        return None, None

    base_model = xgb.XGBRegressor(objective='reg:squarederror', random_state=RANDOM_STATE)
    multi_model = MultiOutputRegressor(base_model, n_jobs=1)
    tscv = TimeSeriesSplit(n_splits=3)
    grid_search = GridSearchCV(estimator=multi_model, param_grid=XGB_PARAM_GRID, cv=tscv,
                               scoring='neg_mean_squared_error', n_jobs=N_JOBS, verbose=2)
    print("  Buscando mejores hiperparámetros...")
    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_
    best_params = {k.replace('estimator__', ''): v for k, v in grid_search.best_params_.items()}
    print(f"  Mejores parámetros: {best_params}")

    val_pred = best_model.predict(X_val)
    val_metrics = compute_metrics(y_val.ravel(), val_pred.ravel())
    print(f"  Métricas en validación: R2={val_metrics['r2']:.3f}, MAE={val_metrics['mae']:.2f}")

    test_pred = best_model.predict(X_test)
    test_metrics = compute_metrics(y_test.ravel(), test_pred.ravel())
    print(f"  Métricas en test: R2={test_metrics['r2']:.3f}, MAE={test_metrics['mae']:.2f}, RMSE={test_metrics['rmse']:.2f}")

    if feature_names is not None and original_csv_path is not None:
        station_metrics = compute_per_station_metrics(y_test, test_pred, X_test, feature_names, output_subdir, entity_name)
    else:
        station_metrics = None

    with open(os.path.join(output_subdir, "model.pkl"), 'wb') as f:
        pickle.dump(best_model, f)
    results = {'best_params': best_params, 'validation_metrics': val_metrics, 'test_metrics': test_metrics,
               'station_metrics': station_metrics, 'n_train': len(X_train), 'n_val': len(X_val), 'n_test': len(X_test)}
    with open(os.path.join(output_subdir, "results.json"), 'w') as f:
        json.dump(results, f, indent=2)

    horizons = [23, 47, 71]
    plot_predictions(y_test, test_pred, horizons, os.path.join(output_subdir, "test_scatter.png"),
                     f"XGBoost - {entity_name} - Test")
    np.save(os.path.join(output_subdir, "test_pred.npy"), test_pred)
    np.save(os.path.join(output_subdir, "test_true.npy"), y_test)
    return best_model, test_metrics


def process_by_transect():
    print("\n" + "="*50)
    print("PROCESANDO XGBOOST POR TRANSECTO")
    ml_2d_dir = os.path.join(DATA_DIR, "by_transect", "ml")
    if not os.path.exists(ml_2d_dir):
        return
    entities = [d for d in os.listdir(ml_2d_dir) if os.path.isdir(os.path.join(ml_2d_dir, d))]
    for entity in entities:
        ml_2d_path = os.path.join(ml_2d_dir, entity, "ml_2d")
        if not os.path.exists(ml_2d_path):
            continue
        X_train = np.load(os.path.join(ml_2d_path, "train_X.npy"))
        y_train = np.load(os.path.join(ml_2d_path, "train_y.npy"))
        X_val   = np.load(os.path.join(ml_2d_path, "val_X.npy"))
        y_val   = np.load(os.path.join(ml_2d_path, "val_y.npy"))
        X_test  = np.load(os.path.join(ml_2d_path, "test_X.npy"))
        y_test  = np.load(os.path.join(ml_2d_path, "test_y.npy"))

        original_csv = os.path.join(ENCODED_DIR, "ml", "by_transect", f"{entity}.csv")
        feature_names = None
        if os.path.exists(original_csv):
            df_cols = pd.read_csv(original_csv, nrows=0, index_col=0)
            feature_names = df_cols.columns.tolist()

        out_subdir = os.path.join(OUTPUT_DIR, "by_transect", entity)
        os.makedirs(out_subdir, exist_ok=True)
        train_and_evaluate_xgb(X_train, y_train, X_val, X_test, y_val, y_test,
                               entity, out_subdir, feature_names, original_csv if os.path.exists(original_csv) else None)


def process_global():
    print("\n" + "="*50)
    print("PROCESANDO XGBOOST GLOBAL")
    ml_2d_dir = os.path.join(DATA_DIR, "global", "ml")
    if not os.path.exists(ml_2d_dir):
        return
    entities = [d for d in os.listdir(ml_2d_dir) if os.path.isdir(os.path.join(ml_2d_dir, d))]
    for entity in entities:
        ml_2d_path = os.path.join(ml_2d_dir, entity, "ml_2d")
        if not os.path.exists(ml_2d_path):
            continue
        X_train = np.load(os.path.join(ml_2d_path, "train_X.npy"))
        y_train = np.load(os.path.join(ml_2d_path, "train_y.npy"))
        X_val   = np.load(os.path.join(ml_2d_path, "val_X.npy"))
        y_val   = np.load(os.path.join(ml_2d_path, "val_y.npy"))
        X_test  = np.load(os.path.join(ml_2d_path, "test_X.npy"))
        y_test  = np.load(os.path.join(ml_2d_path, "test_y.npy"))

        out_subdir = os.path.join(OUTPUT_DIR, "global", entity)
        os.makedirs(out_subdir, exist_ok=True)
        train_and_evaluate_xgb(X_train, y_train, X_val, X_test, y_val, y_test,
                               entity, out_subdir, feature_names=None, original_csv_path=None)


def generate_summary():
    summary = []
    for t in ["by_transect", "global"]:
        dir_path = os.path.join(OUTPUT_DIR, t)
        if not os.path.exists(dir_path):
            continue
        for entity in os.listdir(dir_path):
            res_file = os.path.join(dir_path, entity, "results.json")
            if os.path.exists(res_file):
                with open(res_file, 'r') as f:
                    data = json.load(f)
                summary.append({'entity': entity, 'type': t, **data['test_metrics']})
    if summary:
        pd.DataFrame(summary).to_csv(os.path.join(OUTPUT_DIR, "summary_metrics.csv"), index=False)
        print("\nResumen guardado.")


if __name__ == "__main__":
    print("XGBOOST")
    process_by_transect()
    process_global()
    generate_summary()


#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
RNN (Encoder-Decoder GRU) con KerasTuner.
No entrena si no hay datos de validación.
"""

import os
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GRU, Dense, RepeatVector, TimeDistributed, Reshape, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
import keras_tuner as kt
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

BASE_DIR = os.path.expanduser("/Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/")
WINDOWS_PARTITIONED_DIR = os.path.join(BASE_DIR, "windows_partitioned")
MODELS_DIR = os.path.join(BASE_DIR, "models")
ENCODED_DIR = os.path.join(BASE_DIR, "encoded")  # para cargar CSV originales (necesario para estaciones)

DATA_DIR = WINDOWS_PARTITIONED_DIR
OUTPUT_DIR = os.path.join(MODELS_DIR, "rnn")
os.makedirs(OUTPUT_DIR, exist_ok=True)

WINDOW_IN = 72
WINDOW_OUT = 72
RANDOM_STATE = 42
tf.random.set_seed(RANDOM_STATE)

HP_UNITS = [64, 128]          # elimina 32
HP_DROPOUT = [0.2, 0.3]       # elimina 0.0
HP_LR = [1e-3, 1e-4]          # elimina 5e-4
HP_EPOCHS = 100
BATCH_SIZE = 32


def willmott_index(y_true, y_pred):
    numer = np.sum((y_true - y_pred) ** 2)
    denom = np.sum((np.abs(y_pred - y_true.mean()) + np.abs(y_true - y_true.mean())) ** 2)
    return 1 - numer / denom if denom != 0 else np.nan


def mape(y_true, y_pred):
    mask = y_true != 0
    if not mask.any():
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


def compute_metrics(y_true, y_pred):
    return {
        'r2': r2_score(y_true, y_pred),
        'mae': mean_absolute_error(y_true, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'mape': mape(y_true, y_pred),
        'willmott': willmott_index(y_true, y_pred)
    }


def plot_predictions(y_true, y_pred, horizons, save_path, title):
    n_plots = len(horizons)
    fig, axes = plt.subplots(1, n_plots, figsize=(5*n_plots, 4))
    if n_plots == 1:
        axes = [axes]
    for ax, h in zip(axes, horizons):
        ax.scatter(y_true[:, h], y_pred[:, h], alpha=0.3, s=10)
        ax.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', lw=1)
        ax.set_xlabel('Real O3 (µg/m³)')
        ax.set_ylabel('Predicho O3 (µg/m³)')
        ax.set_title(f'Horizonte {h+1}h')
        ax.grid(True, alpha=0.3)
    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()


def build_model(hp, input_shape, output_dim=72):
    if isinstance(hp, dict):
        units = hp['units']
        dropout = hp['dropout']
        lr = hp['lr']
    else:
        units = hp.Choice('units', HP_UNITS)
        dropout = hp.Choice('dropout', HP_DROPOUT)
        lr = hp.Choice('lr', HP_LR)
    encoder_inputs = Input(shape=input_shape, name='encoder_input')
    encoder = GRU(units, return_state=True, dropout=dropout, recurrent_dropout=dropout, kernel_regularizer=l2(1e-5))
    _, state_h = encoder(encoder_inputs)
    decoder_repeated = RepeatVector(output_dim)(state_h)
    decoder_gru = GRU(units, return_sequences=True, dropout=dropout, recurrent_dropout=dropout, kernel_regularizer=l2(1e-5))
    decoder_outputs = decoder_gru(decoder_repeated, initial_state=state_h)
    decoder_dense = TimeDistributed(Dense(1))(decoder_outputs)
    outputs = Reshape((output_dim,))(decoder_dense)
    model = Model(encoder_inputs, outputs)
    model.compile(optimizer=Adam(learning_rate=lr), loss='mse', metrics=['mae'])
    return model


def get_station_idx_and_mapping(entity_name, transect=True):
    if not transect:
        return None, None
    csv_path = os.path.join(ENCODED_DIR, "dl", "by_transect", f"{entity_name}.csv")
    mapping_path = os.path.join(ENCODED_DIR, "dl", "by_transect", f"{entity_name}_mapping.json")
    if not os.path.exists(csv_path):
        return None, None
    df_cols = pd.read_csv(csv_path, nrows=0, index_col=0)
    feature_names = df_cols.columns.tolist()
    if 'Estacion' not in feature_names:
        return None, None
    idx_estacion = feature_names.index('Estacion')
    if os.path.exists(mapping_path):
        with open(mapping_path, 'r') as f:
            mapping_data = json.load(f)
        est_map = mapping_data.get('Estacion', {})
        mapping_estacion = {int(v): k for k, v in est_map.items()}
    else:
        mapping_estacion = None
    return idx_estacion, mapping_estacion


def compute_per_station_metrics_rnn(y_true, y_pred, X_test, idx_estacion, mapping_estacion, output_dir, entity_name):
    station_preds = {}
    for i in range(len(y_true)):
        station_code = int(round(X_test[i, 0, idx_estacion]))
        station_name = mapping_estacion.get(station_code, f"Unknown_{station_code}")
        station_preds.setdefault(station_name, {'true': [], 'pred': []})
        station_preds[station_name]['true'].append(y_true[i])
        station_preds[station_name]['pred'].append(y_pred[i])
    station_metrics = {}
    for station, data in station_preds.items():
        true_stack = np.vstack(data['true'])
        pred_stack = np.vstack(data['pred'])
        metrics = compute_metrics(true_stack.ravel(), pred_stack.ravel())
        station_metrics[station] = metrics
    if station_metrics:
        df = pd.DataFrame(station_metrics).T
        df.index.name = 'station'
        df.to_csv(os.path.join(output_dir, f"{entity_name}_per_station_metrics.csv"))
    return station_metrics


def train_and_evaluate_rnn(X_train, y_train, X_val, X_test, y_val, y_test,
                           scaler_y, entity_name, output_subdir,
                           idx_estacion=None, mapping_estacion=None):
    print(f"\n--- Entrenando RNN para {entity_name} ---")
    if len(X_val) == 0 or len(y_val) == 0:
        print(f"  Saltando {entity_name}: conjunto de validación vacío.")
        return None, None

    input_shape = (X_train.shape[1], X_train.shape[2])
    tuner = kt.RandomSearch(
        hypermodel=lambda hp: build_model(hp, input_shape),
        objective='val_loss',
        max_trials=len(HP_UNITS)*len(HP_DROPOUT)*len(HP_LR),
        executions_per_trial=1,
        directory=output_subdir,
        project_name='tuning',
        overwrite=True
    )
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
    print("  Buscando hiperparámetros...")
    tuner.search(X_train, y_train, validation_data=(X_val, y_val),
                 epochs=HP_EPOCHS, batch_size=BATCH_SIZE, callbacks=[early_stop, reduce_lr], verbose=1)
    best_hp = tuner.get_best_hyperparameters(1)[0]
    best_params = {'units': best_hp.get('units'), 'dropout': best_hp.get('dropout'), 'lr': best_hp.get('lr')}
    print(f"  Mejores parámetros: {best_params}")

    model = build_model(best_params, input_shape)
    X_train_full = np.concatenate([X_train, X_val], axis=0)
    y_train_full = np.concatenate([y_train, y_val], axis=0)
    history = model.fit(X_train_full, y_train_full, validation_split=0.1, epochs=HP_EPOCHS, batch_size=BATCH_SIZE,
                        callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
                                   ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)], verbose=1)

    y_pred_scaled = model.predict(X_test, verbose=0)
    y_test_descaled = scaler_y.inverse_transform(y_test.reshape(-1, 1)).reshape(y_test.shape)
    y_pred_descaled = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).reshape(y_pred_scaled.shape)
    test_metrics = compute_metrics(y_test_descaled.ravel(), y_pred_descaled.ravel())
    print(f"  Métricas test: R2={test_metrics['r2']:.3f}, MAE={test_metrics['mae']:.2f}, RMSE={test_metrics['rmse']:.2f}")

    if idx_estacion is not None and mapping_estacion is not None:
        station_metrics = compute_per_station_metrics_rnn(y_test_descaled, y_pred_descaled, X_test,
                                                          idx_estacion, mapping_estacion, output_subdir, entity_name)
    else:
        station_metrics = None

    model.save(os.path.join(output_subdir, "model.keras"))
    with open(os.path.join(output_subdir, "history.pkl"), 'wb') as f:
        pickle.dump(history.history, f)
    results = {'best_params': best_params, 'test_metrics': test_metrics, 'station_metrics': station_metrics,
               'n_train': len(X_train_full), 'n_test': len(X_test)}
    with open(os.path.join(output_subdir, "results.json"), 'w') as f:
        json.dump(results, f, indent=2)

    horizons = [23, 47, 71]
    plot_predictions(y_test_descaled, y_pred_descaled, horizons,
                     os.path.join(output_subdir, "test_scatter.png"), f"RNN - {entity_name} - Test")
    np.save(os.path.join(output_subdir, "test_pred.npy"), y_pred_descaled)
    np.save(os.path.join(output_subdir, "test_true.npy"), y_test_descaled)
    return model, test_metrics


def process_by_transect():
    print("\n" + "="*50)
    print("PROCESANDO RNN POR TRANSECTO")
    dl_dir = os.path.join(DATA_DIR, "by_transect", "dl")
    if not os.path.exists(dl_dir):
        return
    entities = [d for d in os.listdir(dl_dir) if os.path.isdir(os.path.join(dl_dir, d))]
    for entity in entities:
        entity_path = os.path.join(dl_dir, entity)
        X_train = np.load(os.path.join(entity_path, "train_X.npy"))
        y_train = np.load(os.path.join(entity_path, "train_y.npy"))
        X_val   = np.load(os.path.join(entity_path, "val_X.npy"))
        y_val   = np.load(os.path.join(entity_path, "val_y.npy"))
        X_test  = np.load(os.path.join(entity_path, "test_X.npy"))
        y_test  = np.load(os.path.join(entity_path, "test_y.npy"))
        with open(os.path.join(entity_path, "scaler_y.pkl"), 'rb') as f:
            scaler_y = pickle.load(f)
        idx_estacion, mapping_estacion = get_station_idx_and_mapping(entity, transect=True)
        out_subdir = os.path.join(OUTPUT_DIR, "by_transect", entity)
        os.makedirs(out_subdir, exist_ok=True)
        train_and_evaluate_rnn(X_train, y_train, X_val, X_test, y_val, y_test,
                               scaler_y, entity, out_subdir, idx_estacion, mapping_estacion)


def process_global():
    print("\n" + "="*50)
    print("PROCESANDO RNN GLOBAL")
    dl_dir = os.path.join(DATA_DIR, "global", "dl")
    if not os.path.exists(dl_dir):
        return
    entities = [d for d in os.listdir(dl_dir) if os.path.isdir(os.path.join(dl_dir, d))]
    for entity in entities:
        entity_path = os.path.join(dl_dir, entity)
        X_train = np.load(os.path.join(entity_path, "train_X.npy"))
        y_train = np.load(os.path.join(entity_path, "train_y.npy"))
        X_val   = np.load(os.path.join(entity_path, "val_X.npy"))
        y_val   = np.load(os.path.join(entity_path, "val_y.npy"))
        X_test  = np.load(os.path.join(entity_path, "test_X.npy"))
        y_test  = np.load(os.path.join(entity_path, "test_y.npy"))
        with open(os.path.join(entity_path, "scaler_y.pkl"), 'rb') as f:
            scaler_y = pickle.load(f)
        out_subdir = os.path.join(OUTPUT_DIR, "global", entity)
        os.makedirs(out_subdir, exist_ok=True)
        train_and_evaluate_rnn(X_train, y_train, X_val, X_test, y_val, y_test,
                               scaler_y, entity, out_subdir, None, None)


def generate_summary():
    summary = []
    for t in ["by_transect", "global"]:
        dir_path = os.path.join(OUTPUT_DIR, t)
        if not os.path.exists(dir_path):
            continue
        for entity in os.listdir(dir_path):
            res_file = os.path.join(dir_path, entity, "results.json")
            if os.path.exists(res_file):
                with open(res_file, 'r') as f:
                    data = json.load(f)
                summary.append({'entity': entity, 'type': t, **data['test_metrics']})
    if summary:
        pd.DataFrame(summary).to_csv(os.path.join(OUTPUT_DIR, "summary_metrics.csv"), index=False)
        print("\nResumen guardado.")


if __name__ == "__main__":
    print("RNN (GRU)")
    process_by_transect()
    process_global()
    generate_summary()


#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
LSTM (Encoder-Decoder) con KerasTuner.
Verifica que validación no esté vacía.
"""

import os
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, RepeatVector, TimeDistributed, Reshape, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
import keras_tuner as kt
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

BASE_DIR = os.path.expanduser("/Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/")
WINDOWS_PARTITIONED_DIR = os.path.join(BASE_DIR, "windows_partitioned")
MODELS_DIR = os.path.join(BASE_DIR, "models")
ENCODED_DIR = os.path.join(BASE_DIR, "encoded")  # para cargar CSV originales (necesario para estaciones)

DATA_DIR = WINDOWS_PARTITIONED_DIR
OUTPUT_DIR = os.path.join(MODELS_DIR, "lstm")
os.makedirs(OUTPUT_DIR, exist_ok=True)

WINDOW_IN = 72
WINDOW_OUT = 72
RANDOM_STATE = 42
tf.random.set_seed(RANDOM_STATE)

HP_UNITS = [64, 128]          # elimina 32
HP_DROPOUT = [0.2, 0.3]       # elimina 0.0
HP_LR = [1e-3, 1e-4]          # elimina 5e-4
HP_EPOCHS = 100
BATCH_SIZE = 32


def willmott_index(y_true, y_pred):
    numer = np.sum((y_true - y_pred) ** 2)
    denom = np.sum((np.abs(y_pred - y_true.mean()) + np.abs(y_true - y_true.mean())) ** 2)
    return 1 - numer / denom if denom != 0 else np.nan


def mape(y_true, y_pred):
    mask = y_true != 0
    if not mask.any():
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


def compute_metrics(y_true, y_pred):
    return {
        'r2': r2_score(y_true, y_pred),
        'mae': mean_absolute_error(y_true, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'mape': mape(y_true, y_pred),
        'willmott': willmott_index(y_true, y_pred)
    }


def plot_predictions(y_true, y_pred, horizons, save_path, title):
    n_plots = len(horizons)
    fig, axes = plt.subplots(1, n_plots, figsize=(5*n_plots, 4))
    if n_plots == 1:
        axes = [axes]
    for ax, h in zip(axes, horizons):
        ax.scatter(y_true[:, h], y_pred[:, h], alpha=0.3, s=10)
        ax.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', lw=1)
        ax.set_xlabel('Real O3 (µg/m³)')
        ax.set_ylabel('Predicho O3 (µg/m³)')
        ax.set_title(f'Horizonte {h+1}h')
        ax.grid(True, alpha=0.3)
    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()


def build_model(hp, input_shape, output_dim=72):
    if isinstance(hp, dict):
        units = hp['units']
        dropout = hp['dropout']
        lr = hp['lr']
    else:
        units = hp.Choice('units', HP_UNITS)
        dropout = hp.Choice('dropout', HP_DROPOUT)
        lr = hp.Choice('lr', HP_LR)
    encoder_inputs = Input(shape=input_shape, name='encoder_input')
    encoder = LSTM(units, return_state=True, dropout=dropout, recurrent_dropout=dropout, kernel_regularizer=l2(1e-5))
    _, state_h, state_c = encoder(encoder_inputs)
    encoder_states = [state_h, state_c]
    decoder_repeated = RepeatVector(output_dim)(state_h)
    decoder_lstm = LSTM(units, return_sequences=True, dropout=dropout, recurrent_dropout=dropout, kernel_regularizer=l2(1e-5))
    decoder_outputs = decoder_lstm(decoder_repeated, initial_state=encoder_states)
    decoder_dense = TimeDistributed(Dense(1))(decoder_outputs)
    outputs = Reshape((output_dim,))(decoder_dense)
    model = Model(encoder_inputs, outputs)
    model.compile(optimizer=Adam(learning_rate=lr), loss='mse', metrics=['mae'])
    return model


def get_station_idx_and_mapping(entity_name, transect=True):
    if not transect:
        return None, None
    csv_path = os.path.join(ENCODED_DIR, "dl", "by_transect", f"{entity_name}.csv")
    mapping_path = os.path.join(ENCODED_DIR, "dl", "by_transect", f"{entity_name}_mapping.json")
    if not os.path.exists(csv_path):
        return None, None
    df_cols = pd.read_csv(csv_path, nrows=0, index_col=0)
    feature_names = df_cols.columns.tolist()
    if 'Estacion' not in feature_names:
        return None, None
    idx_estacion = feature_names.index('Estacion')
    if os.path.exists(mapping_path):
        with open(mapping_path, 'r') as f:
            mapping_data = json.load(f)
        est_map = mapping_data.get('Estacion', {})
        mapping_estacion = {int(v): k for k, v in est_map.items()}
    else:
        mapping_estacion = None
    return idx_estacion, mapping_estacion


def compute_per_station_metrics_lstm(y_true, y_pred, X_test, idx_estacion, mapping_estacion, output_dir, entity_name):
    station_preds = {}
    for i in range(len(y_true)):
        station_code = int(round(X_test[i, 0, idx_estacion]))
        station_name = mapping_estacion.get(station_code, f"Unknown_{station_code}")
        station_preds.setdefault(station_name, {'true': [], 'pred': []})
        station_preds[station_name]['true'].append(y_true[i])
        station_preds[station_name]['pred'].append(y_pred[i])
    station_metrics = {}
    for station, data in station_preds.items():
        true_stack = np.vstack(data['true'])
        pred_stack = np.vstack(data['pred'])
        metrics = compute_metrics(true_stack.ravel(), pred_stack.ravel())
        station_metrics[station] = metrics
    if station_metrics:
        df = pd.DataFrame(station_metrics).T
        df.index.name = 'station'
        df.to_csv(os.path.join(output_dir, f"{entity_name}_per_station_metrics.csv"))
    return station_metrics


def train_and_evaluate_lstm(X_train, y_train, X_val, X_test, y_val, y_test,
                            scaler_y, entity_name, output_subdir,
                            idx_estacion=None, mapping_estacion=None):
    print(f"\n--- Entrenando LSTM para {entity_name} ---")
    if len(X_val) == 0 or len(y_val) == 0:
        print(f"  Saltando {entity_name}: conjunto de validación vacío.")
        return None, None

    input_shape = (X_train.shape[1], X_train.shape[2])
    tuner = kt.RandomSearch(
        hypermodel=lambda hp: build_model(hp, input_shape),
        objective='val_loss',
        max_trials=len(HP_UNITS)*len(HP_DROPOUT)*len(HP_LR),
        executions_per_trial=1,
        directory=output_subdir,
        project_name='tuning',
        overwrite=True
    )
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
    print("  Buscando hiperparámetros...")
    tuner.search(X_train, y_train, validation_data=(X_val, y_val),
                 epochs=HP_EPOCHS, batch_size=BATCH_SIZE, callbacks=[early_stop, reduce_lr], verbose=1)
    best_hp = tuner.get_best_hyperparameters(1)[0]
    best_params = {'units': best_hp.get('units'), 'dropout': best_hp.get('dropout'), 'lr': best_hp.get('lr')}
    print(f"  Mejores parámetros: {best_params}")

    model = build_model(best_params, input_shape)
    X_train_full = np.concatenate([X_train, X_val], axis=0)
    y_train_full = np.concatenate([y_train, y_val], axis=0)
    history = model.fit(X_train_full, y_train_full, validation_split=0.1, epochs=HP_EPOCHS, batch_size=BATCH_SIZE,
                        callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
                                   ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)], verbose=1)

    y_pred_scaled = model.predict(X_test, verbose=0)
    y_test_descaled = scaler_y.inverse_transform(y_test.reshape(-1, 1)).reshape(y_test.shape)
    y_pred_descaled = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).reshape(y_pred_scaled.shape)
    test_metrics = compute_metrics(y_test_descaled.ravel(), y_pred_descaled.ravel())
    print(f"  Métricas test: R2={test_metrics['r2']:.3f}, MAE={test_metrics['mae']:.2f}, RMSE={test_metrics['rmse']:.2f}")

    if idx_estacion is not None and mapping_estacion is not None:
        station_metrics = compute_per_station_metrics_lstm(y_test_descaled, y_pred_descaled, X_test,
                                                           idx_estacion, mapping_estacion, output_subdir, entity_name)
    else:
        station_metrics = None

    model.save(os.path.join(output_subdir, "model.keras"))
    with open(os.path.join(output_subdir, "history.pkl"), 'wb') as f:
        pickle.dump(history.history, f)
    results = {'best_params': best_params, 'test_metrics': test_metrics, 'station_metrics': station_metrics,
               'n_train': len(X_train_full), 'n_test': len(X_test)}
    with open(os.path.join(output_subdir, "results.json"), 'w') as f:
        json.dump(results, f, indent=2)

    horizons = [23, 47, 71]
    plot_predictions(y_test_descaled, y_pred_descaled, horizons,
                     os.path.join(output_subdir, "test_scatter.png"), f"LSTM - {entity_name} - Test")
    np.save(os.path.join(output_subdir, "test_pred.npy"), y_pred_descaled)
    np.save(os.path.join(output_subdir, "test_true.npy"), y_test_descaled)
    return model, test_metrics


def process_by_transect():
    print("\n" + "="*50)
    print("PROCESANDO LSTM POR TRANSECTO")
    dl_dir = os.path.join(DATA_DIR, "by_transect", "dl")
    if not os.path.exists(dl_dir):
        return
    entities = [d for d in os.listdir(dl_dir) if os.path.isdir(os.path.join(dl_dir, d))]
    for entity in entities:
        entity_path = os.path.join(dl_dir, entity)
        X_train = np.load(os.path.join(entity_path, "train_X.npy"))
        y_train = np.load(os.path.join(entity_path, "train_y.npy"))
        X_val   = np.load(os.path.join(entity_path, "val_X.npy"))
        y_val   = np.load(os.path.join(entity_path, "val_y.npy"))
        X_test  = np.load(os.path.join(entity_path, "test_X.npy"))
        y_test  = np.load(os.path.join(entity_path, "test_y.npy"))
        with open(os.path.join(entity_path, "scaler_y.pkl"), 'rb') as f:
            scaler_y = pickle.load(f)
        idx_estacion, mapping_estacion = get_station_idx_and_mapping(entity, transect=True)
        out_subdir = os.path.join(OUTPUT_DIR, "by_transect", entity)
        os.makedirs(out_subdir, exist_ok=True)
        train_and_evaluate_lstm(X_train, y_train, X_val, X_test, y_val, y_test,
                                scaler_y, entity, out_subdir, idx_estacion, mapping_estacion)


def process_global():
    print("\n" + "="*50)
    print("PROCESANDO LSTM GLOBAL")
    dl_dir = os.path.join(DATA_DIR, "global", "dl")
    if not os.path.exists(dl_dir):
        return
    entities = [d for d in os.listdir(dl_dir) if os.path.isdir(os.path.join(dl_dir, d))]
    for entity in entities:
        entity_path = os.path.join(dl_dir, entity)
        X_train = np.load(os.path.join(entity_path, "train_X.npy"))
        y_train = np.load(os.path.join(entity_path, "train_y.npy"))
        X_val   = np.load(os.path.join(entity_path, "val_X.npy"))
        y_val   = np.load(os.path.join(entity_path, "val_y.npy"))
        X_test  = np.load(os.path.join(entity_path, "test_X.npy"))
        y_test  = np.load(os.path.join(entity_path, "test_y.npy"))
        with open(os.path.join(entity_path, "scaler_y.pkl"), 'rb') as f:
            scaler_y = pickle.load(f)
        out_subdir = os.path.join(OUTPUT_DIR, "global", entity)
        os.makedirs(out_subdir, exist_ok=True)
        train_and_evaluate_lstm(X_train, y_train, X_val, X_test, y_val, y_test,
                                scaler_y, entity, out_subdir, None, None)


def generate_summary():
    summary = []
    for t in ["by_transect", "global"]:
        dir_path = os.path.join(OUTPUT_DIR, t)
        if not os.path.exists(dir_path):
            continue
        for entity in os.listdir(dir_path):
            res_file = os.path.join(dir_path, entity, "results.json")
            if os.path.exists(res_file):
                with open(res_file, 'r') as f:
                    data = json.load(f)
                summary.append({'entity': entity, 'type': t, **data['test_metrics']})
    if summary:
        pd.DataFrame(summary).to_csv(os.path.join(OUTPUT_DIR, "summary_metrics.csv"), index=False)
        print("\nResumen guardado.")


if __name__ == "__main__":
    print("LSTM")
    process_by_transect()
    process_global()
    generate_summary()







NameError: name 'BASE_DIR' is not defined